# Auto Loader Pipeline: CSV to Delta

## Overview

This notebook demonstrates **Databricks Auto Loader** for incrementally ingesting CSV files into Delta Lake format. Auto Loader (`cloudFiles`) automatically discovers and processes new files as they arrive in cloud storage, making it ideal for building reliable data pipelines.

### What You'll Learn
* How to configure Auto Loader for CSV file ingestion
* Schema inference and evolution handling
* Writing streaming data to Delta Lake
* Checkpoint-based exactly-once processing

### Use Case
Perfect for scenarios where:
* CSV files are continuously dropped into cloud storage
* You need incremental processing (only new files)
* Schema may evolve over time
* Exactly-once processing guarantees are required

### Prerequisites
* Unity Catalog enabled workspace
* Write access to `databricksmayank.bronze` catalog
* CSV files ready to upload to the volume

## Configuration Reference

### Storage Paths
| Path Type | Location |
| --- | --- |
| **Raw CSV Files** | `/Volumes/databricksmayank/bronze/autovol/raw/` |
| **Delta Output** | `/Volumes/databricksmayank/bronze/autovol/destination/data/` |
| **Checkpoint** | `/Volumes/databricksmayank/bronze/autovol/destination/checkpoint/` |

### Auto Loader Options Explained

**`cloudFiles.format`**: `"csv"`
* Specifies the input file format
* Supports: csv, json, parquet, avro, orc, text, binaryFile

**`cloudFiles.schemaLocation`**: Checkpoint directory
* Stores the inferred schema for consistency across runs
* Enables schema evolution tracking

**`cloudFiles.schemaEvolutionMode`**: `"rescue"`
* **rescue**: Captures unexpected columns in `_rescued_data` (recommended)
* **addNewColumns**: Automatically adds new columns to schema
* **failOnNewColumns**: Fails the stream on schema changes
* **none**: No schema evolution handling

### Write Stream Options

**`outputMode`**: `"append"`
* Adds only new records to the Delta table
* Other modes: complete, update

**`trigger`**: `once=True`
* **Batch mode**: Processes all available files then stops
* Ideal for scheduled jobs
* Alternative: `.trigger(processingTime='1 minute')` for continuous streaming

**`checkpointLocation`**
* Ensures exactly-once processing semantics
* Tracks processed files and stream state
* Critical for fault tolerance

## Step 1: Create Unity Catalog Volume

Create a volume in Unity Catalog to store both raw CSV files and processed Delta data. Volumes provide a unified namespace for managing files in cloud storage.

**Note**: Run this cell only once. If the volume already exists, you can skip this step.

## Step 2: Configure Auto Loader Stream

Set up a streaming DataFrame that monitors the raw data directory for new CSV files. Auto Loader will automatically:
* Detect new files as they arrive
* Infer the schema from CSV headers
* Track which files have been processed
* Handle schema changes gracefully

## Step 3: Write Stream to Delta Lake

Start the streaming job to process files and write results to Delta format. The `trigger(once=True)` option processes all available files once, making this ideal for scheduled batch jobs.

**Checkpoint**: Ensures exactly-once processing semantics and tracks progress.

In [0]:
%sql
create volume databricksmayank.bronze.autovol

In [0]:
df = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format","csv")\
    .option("cloudFiles.schemaLocation","/Volumes/databricksmayank/bronze/autovol/destination/checkpoint/")\
    .option("cloudFiles.schemaEvolutionMode","rescue")\
    .load("/Volumes/databricksmayank/bronze/autovol/raw/")

In [0]:
df.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation","/Volumes/databricksmayank/bronze/autovol/destination/checkpoint/")\
    .trigger(once=True)\
    .start("/Volumes/databricksmayank/bronze/autovol/destination/data/")

## Next Steps & Verification

### Verify the Data
```sql
-- Query the Delta table
SELECT * FROM delta.`/Volumes/databricksmayank/bronze/autovol/destination/data/`
LIMIT 10;

-- Check record count
SELECT COUNT(*) as total_records 
FROM delta.`/Volumes/databricksmayank/bronze/autovol/destination/data/`;
```

### Add New Files
To test incremental loading:
1. Upload new CSV files to `/Volumes/databricksmayank/bronze/autovol/raw/`
2. Re-run Cells 7-8 (readStream and writeStream)
3. Auto Loader will process only the new files

### Convert to Continuous Streaming
To run continuously instead of batch mode, replace:
```python
.trigger(once=True)  # Batch mode
```
with:
```python
.trigger(processingTime='1 minute')  # Continuous streaming
```

### Monitoring & Troubleshooting
* Check checkpoint location for processing state
* Review `_rescued_data` column for schema mismatches
* Monitor streaming query status with `spark.streams.active`

### Additional Resources
* [Auto Loader Documentation](https://docs.databricks.com/ingestion/auto-loader/index.html)
* [Delta Lake Best Practices](https://docs.databricks.com/delta/best-practices.html)

## Testing Incremental Processing

This section demonstrates Auto Loader's incremental processing capability. After uploading additional CSV files to the raw directory, re-running the write stream will process only the new files.

**What happens:**
* Auto Loader detects new files in the source directory
* Only unprocessed files are ingested (checkpoint tracks what's been processed)
* New records are appended to the existing Delta table
* Schema evolution is handled automatically via rescue mode

In [0]:
df.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation","/Volumes/databricksmayank/bronze/autovol/destination/checkpoint/")\
    .trigger(once=True)\
    .start("/Volumes/databricksmayank/bronze/autovol/destination/data/")

## Verify Results: Schema Evolution

The cell below reads and displays the processed Delta table. Check for the `_rescued_data` column which captures any unexpected columns from CSV files that don't match the inferred schema.

**Schema Evolution in Action:**
* `_rescued_data` column contains JSON with unexpected columns
* Original schema remains stable
* No pipeline failures from schema changes
* You can parse `_rescued_data` downstream as needed

In [0]:
df = spark.read.format("delta")\
        .load("/Volumes/databricksmayank/bronze/autovol/destination/data/")
display(df)

## Best Practices & Production Considerations

### Performance Optimization

**File Size**
* Target 128MB - 1GB per file for optimal performance
* Too many small files impacts performance
* Use Auto Compaction for Delta tables

**Partitioning**
```python
df.writeStream.format("delta")\
  .partitionBy("date")\  # Partition by date column
  .option("checkpointLocation", checkpoint_path)\
  .start(output_path)
```

**Schema Hints** (for faster inference)
```python
.option("cloudFiles.schemaHints", "order_amount DOUBLE, order_date DATE")
```

### Monitoring & Operations

**Check Stream Status**
```python
for stream in spark.streams.active:
    print(f"Stream: {stream.name}")
    print(f"Status: {stream.status}")
    print(f"Recent progress: {stream.recentProgress}")
```

**Query Processed Files**
```python
# List processed files from checkpoint
dbutils.fs.ls("/Volumes/databricksmayank/bronze/autovol/destination/checkpoint/sources/0/")
```

### Error Handling

**Rescue Data Analysis**
```sql
SELECT _rescued_data, COUNT(*) as rescue_count
FROM delta.`/Volumes/databricksmayank/bronze/autovol/destination/data/`
WHERE _rescued_data IS NOT NULL
GROUP BY _rescued_data;
```

**Common Issues**
* **Checkpoint conflicts**: Don't reuse checkpoint locations across different streams
* **Schema location**: Keep separate from checkpoint location
* **Volume permissions**: Ensure write access to all paths

### Production Deployment

1. **Schedule as Job**: Convert to Databricks Job for production
2. **Alerting**: Set up alerts on stream failures
3. **Cost optimization**: Use `trigger(availableNow=True)` for cost-efficient batch processing
4. **Data quality**: Add validation checks before writing
5. **Backfill**: For reprocessing, clear checkpoint and use new location

### Additional Resources

* [Auto Loader Documentation](https://docs.databricks.com/ingestion/auto-loader/index.html)
* [Delta Lake Best Practices](https://docs.databricks.com/delta/best-practices.html)
* [Structured Streaming Guide](https://docs.databricks.com/structured-streaming/index.html)